## setup

In [ ]:
# import stuff
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

In [ ]:
# load data from folder
folder = Path('../data/processed')

# filtered datasets with mortgage rates
sold = pd.read_csv(folder / 'wk4_5_sold_clean.csv', low_memory = False)

In [ ]:
# sold.head()
sold.shape

In [ ]:
date_cols = ['CloseDate',
             'PurchaseContractDate',
             'ListingContractDate',
             'ContractStatusChangeDate']

sold[date_cols] = sold[date_cols].apply(pd.to_datetime, errors = 'coerce')

## feature engineering

In [ ]:
# ft eng
sold['price_ratio'] = sold['ClosePrice'] / sold['OriginalListPrice']

# normalizes price across sizes
sold['price_per_sqft'] = sold['ClosePrice'] / sold['LivingArea']

# enables time-series analysis
sold['year'] = sold['CloseDate'].dt.year
sold['month'] = sold['CloseDate'].dt.month
# df['YrMo'] = df['CloseDate'].dt.to_period('M')

# captures full price reduction history
sold['close_to_original_list_ratio'] = sold['ClosePrice'] / sold['OriginalListPrice']

# measures time from listing to accepted offer
sold['listing_to_contract_days'] = sold['PurchaseContractDate'] - sold['ListingContractDate']

# measures time from purchase date to close date
sold['contract_to_close_days'] = sold['CloseDate'] - sold['PurchaseContractDate']

# check that engineered columns were created
sold[['price_ratio',
      'price_per_sqft',
      'year',
      'month',
      'close_to_original_list_ratio',
      'listing_to_contract_days',
      'contract_to_close_days'
      ]].head()

## add school districts

In [ ]:
# add school districts
school_gdf = gpd.read_file('../data/school_districts.geojson')

In [ ]:
# filter by unified school district
filtered_school_gdf = school_gdf[school_gdf['DistrictType'] == 'Unified']

# only get districtname col & geometry for merging
filtered_school_gdf = filtered_school_gdf[['DistrictName', 'geometry']]

# check that changes have been made
filtered_school_gdf.head()

In [ ]:
sold_gdf = gpd.GeoDataFrame(
    sold,
    geometry = gpd.points_from_xy(sold['Longitude'], sold['Latitude']),
    crs = 'EPSG:4326'
)

sold_gdf.head()

In [ ]:
# standardize coordinate systems so they match
# otherwise you get a crs mismatch error
if sold_gdf.crs != school_gdf.crs:
    sold_gdf = sold_gdf.to_crs(school_gdf.crs)

In [ ]:
# spatially merge datasets
merged = gpd.sjoin(
    sold_gdf,
    filtered_school_gdf,
    how = 'left',
    predicate = 'within'
)

merged[['Latitude', 'Longitude', 'DistrictName']].head()

In [ ]:
district_null_pct = merged['DistrictName'].isna().sum() / merged['DistrictName'].shape[0]

print('Percentage of missing values in DistrictName column:',
      round(district_null_pct, 4)
      )

## segment analysis

In [ ]:
metrics = ['PropertySubType',
               'CountyOrParish',
               'MLSAreaMajor',
               'ListOfficeName',
               'BuyerOfficeName']

print('SOLD DATASET:')

for metric in metrics:
    print(f'Summary statistics for {metric}:')
    print(sold[metric].describe(), '\n')

In [ ]:
merged.columns

In [ ]:
school_cols = ['ElementarySchool', 'MiddleOrJuniorSchool', 'HighSchool', 'HighSchoolDistrict', 'DistrictName']

print(merged[school_cols].isna().sum(), '\n')
merged[school_cols].head()

In [ ]:
merged.shape